# Online Retail SQL Validation

**Purpose:** Validate the key metrics from the Power BI dashboard directly on transaction-level data using SQL.

This notebook is part of the case dossier **Business Operations & KPI Governance Case**.  
It uses the UCI Online Retail transaction dataset and reproduces selected dashboard KPIs with SQL queries in SQLite.

**What this proves:**
- Data import and type handling
- KPI logic on raw transaction data
- SQL aggregation with `GROUP BY`, `COUNT`, `SUM`, `CASE WHEN`
- Data quality checks
- Independent validation of Power BI dashboard results

## 1. Setup and data import

The CSV file is loaded with `decimal=","` because the source file uses European decimal commas in `UnitPrice`.
Without this step, prices may be interpreted as text and revenue calculations become wrong.

Expected source file name in Colab:

```text
online_retail_working.csv
```

If your file has another name, change the variable `CSV_FILE` below.

In [ ]:
import pandas as pd
import sqlite3

CSV_FILE = "online_retail_working.csv"

df = pd.read_csv(
    CSV_FILE,
    encoding="ISO-8859-1",
    decimal=","
)

print(df.dtypes)
df.head()

## 2. Data preparation

This step recreates the same business logic used in Power BI:

- `Revenue = Quantity * UnitPrice`
- `IsCancellation`: invoice number starts with `C`
- `IsNegativeQuantity`: quantity is below zero
- `HasCustomerID`: customer identifier is available
- `TransactionType`: separates regular sales, cancellations and negative quantities without official cancellation code
- `YearMonth`: month key for time-based aggregation

In [ ]:
# Convert identifiers to text because they are keys, not numeric values for calculation.
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

# Convert CustomerID to a clean text identifier and keep missing IDs as null.
df["CustomerID"] = df["CustomerID"].astype("Int64").astype(str)
df.loc[df["CustomerID"].isin(["<NA>", "nan", "None"]), "CustomerID"] = None

# Parse transaction timestamp.
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

# Core business logic.
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["IsCancellation"] = df["InvoiceNo"].str.startswith("C")
df["IsNegativeQuantity"] = df["Quantity"] < 0
df["HasCustomerID"] = df["CustomerID"].notna()

df["TransactionType"] = df.apply(
    lambda row: "Cancellation"
    if row["IsCancellation"]
    else ("Negative quantity without C" if row["IsNegativeQuantity"] else "Regular sale"),
    axis=1
)

df["InvoiceDateOnly"] = df["InvoiceDate"].dt.date
df["YearMonth"] = df["InvoiceDate"].dt.strftime("%Y-%m")

df[["InvoiceNo", "Quantity", "UnitPrice", "Revenue", "TransactionType", "HasCustomerID", "YearMonth"]].head()

## 3. Load data into SQLite

SQLite is used as a lightweight SQL database inside the notebook.  
This is not a separate SQL Server. It is a temporary in-memory database that allows us to query the prepared transaction table with SQL.

The table is called:

```text
online_retail_working
```

In [ ]:
conn = sqlite3.connect(":memory:")
df.to_sql("online_retail_working", conn, index=False, if_exists="replace")

pd.read_sql_query("""
SELECT COUNT(*) AS row_count
FROM online_retail_working;
""", conn)

## 4. Helper function for SQL checks

This small helper runs a SQL query and returns the result as a DataFrame.  
It keeps the following SQL checks easy to read.

In [ ]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

## SQL Check 1: Monthly Net Revenue

**Purpose:** Validate the monthly Net Revenue trend shown in the Executive Overview dashboard page.

This query aggregates transaction-level revenue by month and counts distinct invoices.

In [ ]:
monthly_net_revenue = run_sql("""
SELECT
    YearMonth,
    ROUND(SUM(Revenue), 2) AS net_revenue,
    COUNT(DISTINCT InvoiceNo) AS orders
FROM online_retail_working
GROUP BY YearMonth
ORDER BY YearMonth;
""")

monthly_net_revenue

**Interpretation:**  
The query confirms the monthly revenue pattern used in the dashboard. Revenue rises strongly toward autumn 2011 and peaks in November 2011. December 2011 should be interpreted carefully because the dataset does not cover a full month.

## SQL Check 2: Top 10 Countries by Net Revenue

**Purpose:** Validate country performance from the Executive Overview page.

This query groups all transaction rows by country and ranks countries by net revenue.

In [ ]:
top_countries = run_sql("""
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS net_revenue,
    COUNT(DISTINCT InvoiceNo) AS orders
FROM online_retail_working
GROUP BY Country
ORDER BY net_revenue DESC
LIMIT 10;
""")

top_countries

**Interpretation:**  
United Kingdom dominates the dataset by revenue and order volume. This confirms that country comparisons need to be interpreted with UK dominance in mind.

## SQL Check 3: Top 10 Customers by Revenue

**Purpose:** Validate the Customer & Revenue Deep Dive page.

This query considers only transactions with a known `CustomerID` and ranks customers by revenue contribution.

In [ ]:
top_customers = run_sql("""
SELECT
    CustomerID,
    ROUND(SUM(Revenue), 2) AS known_customer_revenue,
    COUNT(DISTINCT InvoiceNo) AS orders
FROM online_retail_working
WHERE HasCustomerID = 1
GROUP BY CustomerID
ORDER BY known_customer_revenue DESC
LIMIT 10;
""")

top_customers

**Interpretation:**  
The query shows that customer value is concentrated among a small number of high-revenue customers. It also separates customer analysis from transactions without identifiable CustomerID.

## SQL Check 4: Cancellation Rate by Month

**Purpose:** Validate the cancellation logic and show whether cancellation activity changes over time.

The query counts distinct invoices and distinct cancellation invoices per month. Cancellation invoices are identified by `InvoiceNo` starting with `C`.

In [ ]:
cancellation_rate_by_month = run_sql("""
SELECT
    YearMonth,
    COUNT(DISTINCT InvoiceNo) AS orders,
    COUNT(DISTINCT CASE WHEN IsCancellation = 1 THEN InvoiceNo END) AS cancellation_orders,
    ROUND(
        1.0 * COUNT(DISTINCT CASE WHEN IsCancellation = 1 THEN InvoiceNo END)
        / COUNT(DISTINCT InvoiceNo),
        4
    ) AS cancellation_rate
FROM online_retail_working
GROUP BY YearMonth
ORDER BY YearMonth;
""")

cancellation_rate_by_month

**Interpretation:**  
The query confirms that cancellations are not an isolated issue. Cancellation rates are visible across all months and should be considered when comparing gross and net revenue.

## SQL Check 5: CustomerID Data Quality

**Purpose:** Validate the data quality issue around missing CustomerIDs.

The query compares transactions with and without CustomerID and shows how many rows, orders and revenue are affected.

In [ ]:
customerid_data_quality = run_sql("""
SELECT
    HasCustomerID,
    COUNT(*) AS rows_count,
    COUNT(DISTINCT InvoiceNo) AS orders,
    ROUND(SUM(Revenue), 2) AS net_revenue
FROM online_retail_working
GROUP BY HasCustomerID;
""")

customerid_data_quality

**Interpretation:**  
A relevant part of the transaction rows has no CustomerID. Those records can still be used for revenue, product, country and time analysis, but they limit customer-level analysis such as customer value, repeat behavior and customer-specific cancellation risk.

## SQL Check 6: Product Cancellation Risk

**Purpose:** Validate the Product & Cancellation Risk dashboard page.

This query identifies product or revenue lines with high cancellation value.  
Cancellation quantities and values are converted to positive values with `ABS()` to show the magnitude of the cancellation effect.

In [ ]:
product_cancellation_risk = run_sql("""
SELECT
    StockCode,
    Description,
    ROUND(SUM(Revenue), 2) AS net_revenue,
    SUM(CASE WHEN TransactionType = 'Regular sale' THEN Quantity ELSE 0 END) AS sold_quantity,
    ABS(SUM(CASE WHEN TransactionType = 'Cancellation' THEN Quantity ELSE 0 END)) AS cancellation_quantity,
    ROUND(ABS(SUM(CASE WHEN TransactionType = 'Cancellation' THEN Revenue ELSE 0 END)), 2) AS cancellation_value,
    ROUND(
        1.0 * ABS(SUM(CASE WHEN TransactionType = 'Cancellation' THEN Quantity ELSE 0 END))
        / NULLIF(SUM(CASE WHEN TransactionType = 'Regular sale' THEN Quantity ELSE 0 END), 0),
        4
    ) AS cancellation_quantity_rate
FROM online_retail_working
GROUP BY StockCode, Description
HAVING cancellation_value > 0
ORDER BY cancellation_value DESC
LIMIT 10;
""")

product_cancellation_risk

**Interpretation:**  
The query identifies product and non-product revenue lines where cancellations have a large absolute value. Some lines show high cancellation values despite low or neutral net revenue, which suggests full reversals, corrections or special booking cases that should be validated with business context.

## 5. Export selected SQL result tables

These exports are optional. They can be used as clean evidence tables for the PDF dossier.

In [ ]:
monthly_net_revenue.to_csv("sql_check_01_monthly_net_revenue.csv", index=False)
top_countries.to_csv("sql_check_02_top_countries.csv", index=False)
top_customers.to_csv("sql_check_03_top_customers.csv", index=False)
cancellation_rate_by_month.to_csv("sql_check_04_cancellation_rate_by_month.csv", index=False)
customerid_data_quality.to_csv("sql_check_05_customerid_data_quality.csv", index=False)
product_cancellation_risk.to_csv("sql_check_06_product_cancellation_risk.csv", index=False)

print("CSV exports created.")

## Summary for the case dossier

The SQL checks validate the Power BI dashboard from the transaction level. They confirm the main findings:

1. Net Revenue rises strongly toward autumn 2011 and peaks in November 2011.
2. United Kingdom dominates country-level revenue.
3. Known customer revenue is concentrated among a small number of high-value customers.
4. Cancellations are visible across the full period and materially affect the difference between gross and net revenue.
5. Missing CustomerIDs limit customer-specific analysis, although the affected records remain useful for revenue, country, product and time-based analysis.
6. Some product or revenue lines show high cancellation values and require operational/business validation.